### Calculation of the Variance on the entire dataset and comparing with the Fed approach

In [ ]:
import anndata as ad
import numpy as np
import os
import json
from typing import List, Tuple
import random
SEED = 55
random.seed(SEED)
np.random.seed(SEED)

def centralized_hvg_calculation(adata_path: str, num_hvg: int = 2000):
    """
    Performs a centralized calculation of variance for all genes and
    selects the top highly variable genes.
    """
    print("Loading the full AnnData dataset...")
    adata = ad.read_h5ad(adata_path)
    
    # Extract the expression matrix
    X_matrix = adata.X
    
    print("Calculating gene-wise means and mean of squares...")
    # .mean(axis=0) computes the mean for each column (gene)
    # The toarray() call handles both sparse and dense matrices
    gene_means = np.mean(X_matrix.toarray(), axis=0) if hasattr(X_matrix, 'toarray') else np.mean(X_matrix, axis=0)
    gene_mean_of_squares = np.mean((X_matrix**2).toarray(), axis=0) if hasattr(X_matrix, 'toarray') else np.mean(X_matrix**2, axis=0)
    
    # Calculate variance using the formula: Var(X) = E[X^2] - (E[X])^2
    print("Calculating gene-wise variance...")
    gene_variances = gene_mean_of_squares - (gene_means ** 2)
    
    # Get the global gene list
    global_gene_list = adata.var_names.tolist()

    # Select the top N genes
    print(f"Selecting the top {num_hvg} highly variable genes...")
    # Get the indices that would sort the variances in descending order
    sorted_indices = np.argsort(gene_variances)[::-1]
    
    # Select the top num_hvg indices
    top_hvg_indices = sorted_indices[:num_hvg]
    
    # Get the names of the top HVGs
    top_hvg_names = [global_gene_list[i] for i in top_hvg_indices]
    
    # Sort the final indices for consistent output
    sorted_top_hvg_indices = sorted(top_hvg_indices)

    print(f"Calculation complete. Found {len(top_hvg_names)} HVGs.")
    print("Top 10 HVGs (by variance):", top_hvg_names[:10])
    
    return top_hvg_names, sorted_top_hvg_indices


DATA_PATH = "../0_data/pancreas_train.h5ad"
# Ensure the data file exists before running
if not os.path.exists(DATA_PATH):
    print(f"Error: Dataset not found at {DATA_PATH}. Please provide the correct path.")
else:
    hvg_names, hvg_indices = centralized_hvg_calculation(DATA_PATH, num_hvg=2000)
    hvg_indices = sorted([int(i) for i in hvg_indices])
    
    # Save results for comparison with the FL approach
    import json
    with open("hvg_centralized_names.json", "w") as f:
        json.dump(hvg_names, f)
    with open("hvg_centralized_indices.json", "w") as f:
        json.dump(hvg_indices, f)

    print(f"Saved centralized HVG names to hvg_centralized_names.json and indices to hvg_centralized_indices.json")

Loading the full AnnData dataset...
Calculating gene-wise means and mean of squares...
Calculating gene-wise variance...
Selecting the top 2000 highly variable genes...
Calculation complete. Found 2000 HVGs.
Top 10 HVGs (by variance): ['GCG', 'TTR', 'INS', 'IAPP', 'REG1A', 'SST', 'CTRB2', 'CTRB1', 'CELA3A', 'SPINK1']
Saved centralized HVG names to hvg_centralized_names.json and indices to hvg_centralized_indices.json


In [2]:
# Load the federated HVG list
with open("./hvg_list.json") as fed_file:
    fed_hvg_names = set(json.load(fed_file))

# Convert centralized HVG names to set for comparison
centralized_hvg_names = set(hvg_names)

# Calculate overlap and differences
overlap = centralized_hvg_names & fed_hvg_names
only_centralized = centralized_hvg_names - fed_hvg_names
only_federated = fed_hvg_names - centralized_hvg_names

print(f"Number of overlapping HVGs: {len(overlap)}")
print(f"Number of HVGs only in centralized: {len(only_centralized)}")
print(f"Number of HVGs only in federated: {len(only_federated)}")
print(f"Jaccard similarity: {len(overlap) / len(centralized_hvg_names | fed_hvg_names):.4f}")

Number of overlapping HVGs: 2000
Number of HVGs only in centralized: 0
Number of HVGs only in federated: 0
Jaccard similarity: 1.0000
